<a href="https://colab.research.google.com/github/sandeepProject/Online-Book-Popularity/blob/main/notebooks/Amazon_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Amazon Web Scraping

Working Code

In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import random

def scrape_books(pages_to_scrape=2):
    results = []
    seen_isbns = set()

    HEADERS = {

        #"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US, en;q=0.5",
        "Referer": "https://www.google.com/"
    }

    session = requests.Session()

    for p in range(1, pages_to_scrape + 1):
        print(f"\n--- Scraping Page {p} ---")
        #URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&page={p}"
        #URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&rh=n%3A976389031%2Cn%3A1318216031&dc&page={p}"
        #URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&rh=n%3A976389031%2Cn%3A1318157031&dc&page={p}"
        #URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&rh=n%3A976389031%2Cn%3A1318070031&dc&page={p}"
        #URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&rh=n%3A976389031%2Cn%3A1318157031&dc&page={p}"
        URL = f"https://www.amazon.in/s?k=Books&i=stripbooks&rh=n%3A976389031%2Cn%3A23033695031&dc&page={p}"
        try:
            time.sleep(random.uniform(4, 8))
            response = session.get(URL, headers=HEADERS, timeout=20)
            #print("hai1")

            if response.status_code == 503 or "api-services-support@amazon.com" in response.text:
                print("Blocked by Amazon (Bot Detection). Try again later or use a VPN.")
                break

            soup = BeautifulSoup(response.content, "html.parser")
            # Target the main search result containers
            book_containers = soup.find_all("div", {"data-component-type": "s-search-result"})
            #print("H1")
            #print(len(book_containers))

            for container in book_containers:
                link_tag = container.find("a", class_="a-link-normal s-no-outline")
                if not link_tag: continue
                #print('H2')

                href = link_tag.get('href')
                # FIX: Check if the link is relative or absolute
                if href.startswith('http'):
                    product_url = href
                else:
                    product_url = 'https://www.amazon.in' + href

                #print("hai2")

                # Randomized delay to prevent 503 errors
                time.sleep(random.uniform(2, 7))

                try:
                    product_page = session.get(product_url, headers=HEADERS, timeout=10)
                    psoup = BeautifulSoup(product_page.content, "html.parser")
                    #print("hai3")

                    # --- EXTRACTION ---

                    # 1. Title
                    title = psoup.find("span", id="productTitle")
                    title = title.get_text(strip=True) if title else "N/A"

                    # 2. ISBN-13 (Look inside the detail list)
                    isbn13 = "N/A"
                    for li in psoup.select("#detailBullets_feature_div li"):
                        text = li.get_text()
                        if "ISBN-13" in text:
                            isbn13 = re.search(r'[\d-]+', text.split(':')[-1]).group()


                    # Duplicate Check
                    if isbn13 != "N/A" and isbn13 in seen_isbns:
                        continue

                    # 3. Author
                    author = psoup.find("span", class_="author")
                    author = author.find("a").get_text(strip=True) if author and author.find("a") else "N/A"

                    # 4. Price
                    price = psoup.find("span", class_="a-price-whole")
                    price = price.get_text(strip=True) if price else "N/A"

                    # 5. Rating & Reviews
                    rating = psoup.find("span", {"data-hook": "rating-out-of-text"})
                    rating = rating.get_text(strip=True).split(" ")[0] if rating else "N/A"

                    rcount = psoup.find("span", id="acrCustomerReviewText")
                    rcount = "".join(filter(str.isdigit, rcount.get_text())) if rcount else "0"

                    # 6. Extract Format (Binding type) and Published Date
                    subtitle_tag = psoup.find("span", id="productSubtitle")
                    if subtitle_tag:
                      subtitle_text = subtitle_tag.get_text(strip=True)
                      # Split by the dash (Amazon often uses the en-dash '–' or hyphen '-')
                      if "–" in subtitle_text:
                        parts = subtitle_text.split("–")
                      elif "-" in subtitle_text:
                        parts = subtitle_text.split("-")
                      else:
                        parts = [subtitle_text, "N/A"]

                      book_format = parts[0].strip() # "Binding Type"
                      publish_date = parts[1].strip() if len(parts) > 1 else "N/A" # "Published Date"
                    else:
                      book_format = "N/A"
                      publish_date = "N/A"

                    # 7. Extract Publication Date2
                    pub_date_tag = psoup.find("span", class_="a-text-bold", string=lambda t: t and "Publication date" in t)

                    if pub_date_tag:
                        # Jump to the next span containing '20 November 2017'
                        publication_date = pub_date_tag.find_next("span").get_text(strip=True)
                    else:
                        publication_date = "N/A"

                    # 8. Extract Pages
                    pages_tag = psoup.find("span", class_="a-text-bold", string=lambda t: t and "Print length" in t)
                    if pages_tag:
                    # Jump to the next span which contains the actual value
                      print_length = pages_tag.find_next("span").get_text(strip=True).replace('pages',"") if pages_tag else "N/A"

                    # 9. Extract Language
                    lang_tag = psoup.find("span", class_="a-text-bold", string=lambda t: t and "Language" in t)
                    if lang_tag:
                    # Jump to the next span which contains the actual value
                      lang = lang_tag.find_next("span").get_text(strip=True) if lang_tag else "N/A"

                    # 10. M.R.P.
                    mrp_tag = psoup.find("span", class_="a-text-price")
                    mrp = mrp_tag.find("span", class_="a-offscreen").get_text(strip=True) if mrp_tag else "N/A"

                    # 11. Extract Rank List and Genre
                    rank_list = psoup.find("ul", class_="zg_hrsr")


                    if rank_list:

                        # Get the first category ranking (e.g., #1 in Cake Making Supplies)
                        first_rank_item = rank_list.find("li")

                        if first_rank_item:
                            full_text = first_rank_item.get_text(strip=True) # Result: "#1 in Cake Making Supplies"
                            # print(full_text)
                            # import re
                            match = re.search(r"#(\d+)\s*in\s*(.*?)(?:\s*\(Books\))?$", full_text)
                            # print(match)
                            if match:
                              seller_rank = match.group(1)   # Result: '1'
                              genre = match.group(2).strip() # Result: 'Horror'
                            else:
                              # Fallback if regex fails
                              seller_rank = "N/A"
                              genre = "N/A"
                    else:
                        seller_rank, genre = "N/A", "N/A"

                    # 12. Extract Star Rating Percentage
                    # Initialize a dictionary with defaults to prevent errors
                    star_data = {
                        '5-Star %': '0%',
                        '4-Star %': '0%',
                        '3-Star %': '0%',
                        '2-Star %': '0%',
                        '1-Star %': '0%'
                    }

                    # Mapping Amazon's internal IDs to our dictionary keys
                    mapping = {
                        "customerReviews-histogram-fiveStar": "5-Star %",
                        "customerReviews-histogram-fourStar": "4-Star %",
                        "customerReviews-histogram-threeStar": "3-Star %",
                        "customerReviews-histogram-twoStar": "2-Star %",
                        "customerReviews-histogram-oneStar": "1-Star %"
                    }

                    for content_id, key in mapping.items():
                        # Find the specific div for the star level
                        star_div = psoup.find("div", {"data-csa-c-content-id": content_id})

                        if star_div:
                            link = star_div.find("a")
                            if link and "aria-label" in link.attrs:
                                # label looks like: "31 percent of reviews have 4 stars"
                                label = link["aria-label"]
                                # Extract the first part (the number) and add '%'
                                star_data[key] = label.split(" ")[0] + "%"

                    # --- SAVE DATA ---
                    if title != "N/A":
                        if isbn13 != "N/A": seen_isbns.add(isbn13)
                        results.append({
                            'ISBN-13': isbn13,
                            'Title': title,
                            'Author': author,
                            'Sales Price': price,
                            'MRP': mrp,
                            'Rating': rating,
                            'Review Count': rcount,
                            'Binding': book_format,
                            'Published Date': publish_date,
                            'Publication Date': publication_date,
                            'Pages': print_length,
                            'Language': lang,
                            'Best Sellers Rank': seller_rank,
                            'Genre': genre,
                            '5-Star %': star_data['5-Star %'],
                            '4-Star %': star_data['4-Star %'],
                            '3-Star %': star_data['3-Star %'],
                            '2-Star %': star_data['2-Star %'],
                            '1-Star %': star_data['1-Star %']

                            # 'URL': product_url[:50] # Shortened for display
                        })
                        print(f"Scraped: {title[:40]}...")

                except Exception as e:
                    print(f"Error on product: {e}")

        except Exception as e:
            print(f"Page error: {e}")
            break

    return pd.DataFrame(results)

df = scrape_books(pages_to_scrape=30) # Start with 1 page to test
print(df.head())


--- Scraping Page 1 ---
Scraped: You Can by George Matthew Adams | The Cl...
Scraped: The Art of Letting Go: Move Beyond the H...
Scraped: Courage To Be Disliked, The: How to free...
Scraped: Ikigai: The Japanese secret to a long an...
Scraped: Don't Believe Everything You Think (Engl...
Scraped: World’s Greatest Books For Personal Grow...
Scraped: Atomic Habits: Tiny Changes, Remarkable ...
Scraped: Manifest Anything in 100 Days: Manifesta...
Scraped: Stop Letting Everything Affect You: How ...
Scraped: Think and Grow Rich by Napoleon Hill - S...
Scraped: The Power of Your Subconscious Mind: Ori...
Scraped: The Book of Five Rings: The Strategy of ...
Scraped: READ PEOPLE LIKE A BOOK | Master Human B...
Scraped: Dopamine Detox: A Short Guide to Remove ...
Scraped: The Art of Being Alone: Loneliness Was M...
Scraped: How to Talk to Anyone: 92 Little Tricks ...

--- Scraping Page 2 ---
Scraped: The Practicing Mind: Developing Focus An...
Scraped: You Only Live Once: A Journey of Self-Di

In [18]:
df.shape

(336, 19)

In [19]:
# (This counts every occurrence AFTER the first one)
duplicate_count = df.duplicated(subset=['ISBN-13','Title'], keep=False).sum()

print(f"Number of duplicate titles: {duplicate_count}")

# 2. To see which specific titles are duplicated and how many times:
print("\nDuplicate Breakdown:")
print(df['Title'].value_counts()[df['Title'].value_counts() > 1])
print(df['ISBN-13'].value_counts()[df['ISBN-13'].value_counts() > 1])

Number of duplicate titles: 2

Duplicate Breakdown:
Title
Read People Like a Book                                  2
Quantum Biology: Healing the Mind With Secret Science    2
Name: count, dtype: int64
ISBN-13
N/A    31
Name: count, dtype: int64


In [20]:
df.to_csv('amazon_data336_16.csv', index=False, encoding='utf-8-sig')

print("Data successfully saved to amazon_data.csv")

Data successfully saved to amazon_data.csv


In [ ]:
df1 = pd.read_csv('amazon_data313_1.csv')
df2 = pd.read_csv('amazon_data161_2.csv')
df3 = pd.read_csv('amazon_data155_3.csv')
df4 = pd.read_csv('amazon_data163_4.csv')
df5 = pd.read_csv('amazon_data145_5.csv')
df6 = pd.read_csv('amazon_data152_6.csv')
df7 = pd.read_csv('amazon_data85_7.csv')
df8 = pd.read_csv('amazon_data11_8.csv')
df9 = pd.read_csv('amazon_data160_9.csv')
df10 = pd.read_csv('amazon_data149_10.csv')



In [ ]:
# 1. Put all your dataframes into a list
dataframes = [df1, df2, df3, df4, df5, df6, df7, df8, df9, df10]

# 2. Concatenate them into one single dataframe
final_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
final_df.shape

(1494, 19)

In [ ]:
final_df.to_csv('amazon_data_final1.csv', index=False, encoding='utf-8-sig')

In [21]:
df11 = pd.read_csv('amazon_data313_11.csv')
df12 = pd.read_csv('amazon_data320_12.csv')
df13 = pd.read_csv('amazon_data310_13.csv')
df14 = pd.read_csv('amazon_data319_14.csv')
df15 = pd.read_csv('amazon_data336_15.csv')
df16 = pd.read_csv('amazon_data336_16.csv')

In [22]:
# 1. Put all your dataframes into a list
dataframes = [df11, df12, df13, df14, df15, df16]

# 2. Concatenate them into one single dataframe
final_df = pd.concat(dataframes, ignore_index=True)

In [23]:
final_df.shape

(1934, 19)

In [24]:
final_df.to_csv('amazon_data_final2.csv', index=False, encoding='utf-8-sig')

In [25]:
df11 = pd.read_csv('amazon_data_final1.csv')
df22 = pd.read_csv('amazon_data_final2.csv')

# 1. Put all your dataframes into a list
dataframes = [df11, df22]

# 2. Concatenate them into one single dataframe
final_df = pd.concat(dataframes, ignore_index=True)

In [26]:
final_df.shape

(3428, 19)

In [27]:
final_df.to_csv('amazon_data_core.csv', index=False, encoding='utf-8-sig')